# Phenotype data imputation



## Description

Empirical Bayes Matrix Factorization (EBMF) is the primary imputation method we use to impute molecular phenotype data [[cf. Qi et al., medRxiv,  2023](https://doi.org/10.1101/2023.11.29.23299181)]. We also provide a collection of other methods to impute missing omics data values including missing forest, XGboost, k-nearest neighbors, soft impute, mean imputation, and limit of detection.

## Input
* A molecular phenotype data with missing where first four columns are chr, start, end, and ID. The rest columns are samples.

## Output
* A complete molecular phenotype data where first four columns are chr, start, end, and ID. The rest columns are samples.

## Minimal Working Example Steps


## ii. Imputation

Timing < 10 min

FIXME - get output

In [ ]:
sos run pipeline/phenotype_imputation.ipynb gEBMF \
    --phenoFile data/protocol_example.protein.bed.gz \
    --cwd output/phenotype/impute_gebmf \
    --no-qc-prior-to-impute 

```

```

## Troubleshooting

| Step | Substep | Problem | Possible Reason | Solution |
|------|---------|---------|------------------|---------|
|  |  |  |  |  |




## Command Interface

In [ ]:
sos run phenotype_imputation.ipynb -h

```
usage: sos run phenotype_imputation.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  targets:              One or more targets to generate
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters

Workflows:
  EBMF
  gEBMF
  missforest
  missxgboost
  knn
  soft
  mean
  lod
  bed_filter_na

Global Workflow Options:
  --cwd output (as path)
                        Work directory & output directory
  --phenoFile VAL (as path, required)
                        Molecular phenotype matrix
  --[no-]qc-prior-to-impute (default to True)
                        QC before imputation to remove unqualified features
  --job-size 1 (as int)
                        For cluster jobs, number commands to run per job
  --walltime 72h
                        Wall clock time expected
  --mem 16G
                        Memory expected
  --numThreads 20 (as int)
                        Number of threads
  --container ''
  --entrypoint ''

Sections
  EBMF:
    Workflow Options:
      --prior 'ebnm_point_laplace'
                        prior distribution of loadings and factors
      --varType 1
      --num-factor 60
  gEBMF:
    Workflow Options:
      --nCores 1
      --num-factor 60
      --backfit-iter 3
      --[no-]save-flash (default to True)
      --[no-]null-check (default to False)
  missforest:
  missxgboost:
  knn:
  soft:
  mean:
  lod:
  bed_filter_na:
    Workflow Options:
      --rank-max 50 (as int)
      --lambda-hyp 30 (as int)
      --impute-method soft
      --tol-missing 0.05 (as float)
                        Tolerance of missingness rows with missing rate larger than tol_missing will be removed, with missing rate smaller than tol_missing will be mean_imputed. Say if
                        we want to keep rows with less than 5% missing, then we use 0.05 as tol_missing.
```

## Setup and global parameters

In [ ]:
[global]
parameter: renovated_code_dir = path('renovated_code')  # override with --renovated-code-dir
# Work directory & output directory
parameter: cwd = path("output")
# Molecular phenotype matrix
parameter: phenoFile = path
# QC before imputation to remove unqualified features 
parameter: qc_prior_to_impute = True
# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "72h"
# Memory expected
parameter: mem = "16G"
# Number of threads
parameter: numThreads = 20
parameter: container = ""
import re
parameter: entrypoint= ""

In [ ]:
[EBMF]
# prior distribution of loadings and factors
parameter: prior = "ebnm_point_laplace"
parameter: varType = '1'
parameter: num_factor = '60'
input: phenoFile
output: f'{cwd:a}/{_input:bn}.imputed.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}', cores = numThreads  
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/data_preprocessing/phenotype/phenotype_imputation.R \
        --step EBMF \
        --cwd "${cwd}" \
        --phenoFile "${_input}" \
        ${"--qc-prior-to-impute" if qc_prior_to_impute else ""} \
        --prior "${prior}" \
        --varType "${varType}" \
        --num-factor ${num_factor} \
        --numThreads ${numThreads}


In [ ]:
[gEBMF]
parameter: nCores = '1'
parameter: num_factor = '60'
parameter: backfit_iter = '3'
parameter: save_flash = True
parameter: null_check = False
input: phenoFile
output: f'{cwd:a}/{_input:bn}.imputed.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}', cores = numThreads  
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/data_preprocessing/phenotype/phenotype_imputation.R \
        --step gEBMF \
        --cwd "${cwd}" \
        --phenoFile "${_input}" \
        ${"--qc-prior-to-impute" if qc_prior_to_impute else ""} \
        --num-factor ${num_factor} \
        --nCores ${nCores} \
        --backfit-iter ${backfit_iter} \
        ${"--save-flash" if save_flash else ""} \
        ${"--null-check" if null_check else ""} \
        --numThreads ${numThreads}


In [ ]:
[missforest]
input: phenoFile
output: f'{cwd:a}/{_input:bn}.imputed.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'  
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/data_preprocessing/phenotype/phenotype_imputation.R \
        --step missforest \
        --cwd "${cwd}" \
        --phenoFile "${_input}" \
        ${"--qc-prior-to-impute" if qc_prior_to_impute else ""} \
        --numThreads ${numThreads}


In [ ]:
[missxgboost]
input: phenoFile
output: f'{cwd:a}/{_input:bn}.imputed.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'  
R: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint=entrypoint
   source('./xgb_imp.R')
   library("tibble")
   library("readr")
   library("dplyr")
   library("missForest")

    pheno <- read_delim(${_input:ar}, delim = "\t")
    # Get index of features that have > 40% missing 
    miss.ind <- which(rowSums(is.na(pheno[, 5:ncol(pheno)]))/(ncol(pheno) -4) > 0.4) 
    # Get index of features where more than 95% values are zero.
    value0.ind <- which(rowSums((pheno[, 5:ncol(pheno)] == 0 | is.na(pheno[, 5:ncol(pheno)])))/(ncol(pheno) -4) > 0.95)
    miss.ind <- c(miss.ind, value0.ind)
    # remove these rows if qc_prior_to_impute = True
    if (${"TRUE" if qc_prior_to_impute else "FALSE"}) {
      if (length(miss.ind) > 0) {
         pheno_qc <- pheno[-miss.ind, ]
       } else {
         pheno_qc <- pheno
       }
      pheno_NAs <- pheno_qc[, 5:ncol(pheno_qc)]
    } else {
      pheno_NAs <- pheno[, 5:ncol(pheno)]
    }
    pheno_imp <- xgboost_imputation(as.matrix(pheno_NAs))
    write_delim(cbind(pheno_qc[, 1:4], pheno_imp), "${_output:n}", delim = "\t" )

bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container=container, entrypoint=entrypoint
    bgzip -f ${_output:n}
    tabix ${_output}
bash: expand= "$[ ]", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint=entrypoint
        stdout=$[_output:n].stdout
        for i in $[_output] ; do 
        echo "output_info: $i " >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `cat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_column:" `cat $i | head -1 | wc -w `   >> $stdout;
        echo "output_headerow:" `cat $i | grep "##" | wc -l `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        cat $i  | grep -v "##" | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done

In [ ]:
[knn]
input: phenoFile
output: f'{cwd:a}/{_input:bn}.imputed.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'  
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/data_preprocessing/phenotype/phenotype_imputation.R \
        --step knn \
        --cwd "${cwd}" \
        --phenoFile "${_input}" \
        ${"--qc-prior-to-impute" if qc_prior_to_impute else ""} \
        --numThreads ${numThreads}


In [ ]:
[soft]
input: phenoFile
output: f'{cwd:a}/{_input:bn}.imputed.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'  
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/data_preprocessing/phenotype/phenotype_imputation.R \
        --step soft \
        --cwd "${cwd}" \
        --phenoFile "${_input}" \
        ${"--qc-prior-to-impute" if qc_prior_to_impute else ""} \
        --numThreads ${numThreads}


In [ ]:
[mean]
input: phenoFile
output: f'{cwd:a}/{_input:bn}.imputed.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'  
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/data_preprocessing/phenotype/phenotype_imputation.R \
        --step mean \
        --cwd "${cwd}" \
        --phenoFile "${_input}" \
        ${"--qc-prior-to-impute" if qc_prior_to_impute else ""} \
        --numThreads ${numThreads}


In [ ]:
[lod]
input: phenoFile
output: f'{cwd:a}/{_input:bn}.imputed.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'  
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/data_preprocessing/phenotype/phenotype_imputation.R \
        --step lod \
        --cwd "${cwd}" \
        --phenoFile "${_input}" \
        ${"--qc-prior-to-impute" if qc_prior_to_impute else ""} \
        --numThreads ${numThreads}


In [ ]:
[bed_filter_na]
parameter: rank_max = 50 # max rank estimated in the per-chr methyl matrix
parameter: lambda_hyp = 30 # hyper par, indicating the importance of the nuclear norm
parameter: impute_method = "soft"
# Tolerance of missingness rows with missing rate larger than tol_missing will be removed,
# with missing rate smaller than tol_missing will be mean_imputed. Say if we want to keep rows with less than 5% missing, then we use 0.05 as tol_missing.
parameter: tol_missing = 0.05
input: phenoFile
output: f'{cwd:a}/{_input:nn}.filter_na.{impute_method}.bed.gz'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/data_preprocessing/phenotype/phenotype_imputation.R \
        --step bed_filter_na \
        --cwd "${cwd}" \
        --phenoFile "${_input}" \
        --rank-max ${rank_max} \
        --lambda-hyp ${lambda_hyp} \
        --impute-method "${impute_method}" \
        --tol-missing ${tol_missing} \
        --numThreads ${numThreads}


The resource usage for softimputing 450K methylation data are as followed:

``` 
time elapsed: 880.90s
peak first occurred: 152.11s
peak last occurred: 175.41s
max vms_memory: 38.95GB
max rss_memory: 34.35GB
memory check interval: 1s
return code: 0
```